# 04 · Custom model training & evaluation — §4 [25 marks]

Run the official pass on the **same machine as notebook 03** — training time per epoch is compared across the two.

| | |
|---|---|
| **Owner** | Member 3 |
| **Trains** | `model_a` (Model B's run comes from notebook 03) |
| **Reads** | the committed split; `model_b__adam`'s committed results |
| **Writes (official)** | `model_a` history + test metrics, curves, confusion matrix |
| **Report needs** | loss curves for both models; accuracy, confusion matrix, precision and recall for both; the A-vs-B trade-off discussion (table built in 07) |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
from IPython.display import display

from edgecnn.data import build_dataloaders
from edgecnn.evaluation import evaluate_run
from edgecnn.evaluation.curves import plot_loss_curves
from edgecnn.evaluation.reporting import plot_confusion_matrix
from edgecnn.models import build_model_from_config
from edgecnn.training import Trainer

In [ ]:
# Shared figure style (Member 1): small inline figures, print-quality saved PNGs.
from edgecnn.evaluation.plotting import apply_style

apply_style()

## Config

In [ ]:
cfg = load_config("configs/experiments/model_a__adam.yaml", mode=MODE)
print(f"{cfg.run_id}  mode={cfg.mode}  official={cfg.is_official}")

## 1 · Data

In [ ]:
data = build_dataloaders(cfg)
print(data.describe())

## 2 · Build Model A

In [ ]:
model = build_model_from_config(cfg, data.num_classes)
print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")

## 3 · Train

In [ ]:
result = Trainer(cfg).fit(model, data, cfg)

## 4 · Loss curves

The train/val gap is the overfitting signal; the vertical line marks the checkpoint that gets evaluated.

In [ ]:
plot_loss_curves(result, write=cfg.is_official)

## 5 · Test-set evaluation

In [ ]:
evaluation = evaluate_run(cfg, data, result.checkpoint_path)
print(f"accuracy {evaluation.accuracy:.4f} | macro precision {evaluation.macro_precision:.4f} "
      f"| macro recall {evaluation.macro_recall:.4f}")

## 6 · Confusion matrix

Normalised by true class, so the diagonal reads as per-class recall.

In [ ]:
plot_confusion_matrix(evaluation, write=cfg.is_official)

## 7 · Model B, from notebook 03

Model B's official run is `model_b__adam` in notebook 03. This notebook only reads its committed results, so B is never trained twice.

In [ ]:
RUN_B = load_config("configs/experiments/model_b__adam.yaml").run_id
if paths.test_metrics_json(RUN_B).exists():
    display(plot_loss_curves(RUN_B))
    display(plot_confusion_matrix(RUN_B))
else:
    print(f"{RUN_B} has no committed results yet - run notebook 03 in official mode first.")

## 8 · Side by side — preview

The official A-vs-B table is built in notebook 07.

In [ ]:
if paths.test_metrics_json(RUN_B).exists():
    b = schema.read_json(paths.test_metrics_json(RUN_B), "test_metrics")
    print(f"test accuracy   Model A {evaluation.accuracy:.4f}   Model B {b['accuracy']:.4f}")

## Discussion prompts — §4

- What did moving from standard to depthwise-separable convolutions cost in accuracy?
- What did it buy — parameters, size on disk, MACs, time per epoch? State ratios.
- Which classes does each model confuse, and why might that be?

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] run on the same machine / runtime as notebook 03's official run
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Guide: COLAB.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.metrics_dir(cfg.run_id), paths.figures_dir(cfg.run_id)],
        "Add model_a results (Colab)",
    )